# Brain Tumor Detection Experiment Tracking
This notebook helps track dataset layout, model performance, and evaluation metrics for the brain tumor detection project.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from src.utils import load_history_npz, load_metrics_json

DATASET_ROOT = Path('dataset')
ARTIFACTS_ROOT = Path('artifacts')
MODEL_TYPES = ['cnn', 'transfer', 'vit']


def inspect_dataset_structure(dataset_root):
    root = Path(dataset_root)
    layout = {}
    for split in ['train', 'val', 'test']:
        split_dir = root / split
        if not split_dir.exists():
            layout[split] = None
            continue
        counts = {}
        for class_dir in sorted(split_dir.iterdir()):
            if class_dir.is_dir():
                counts[class_dir.name] = sum(1 for _ in class_dir.glob('*') if _.is_file())
        layout[split] = counts
    return layout


def build_comparison_frame(artifacts_root):
    rows = []
    for model_type in MODEL_TYPES:
        metrics_file = artifacts_root / f'{model_type}_evaluation_metrics.json'
        if metrics_file.exists():
            metrics = load_metrics_json(metrics_file)
            report = metrics.get('classification_report', {})
            weighted = report.get('weighted avg', report.get('weighted_avg', {})) if isinstance(report, dict) else {}
            rows.append({
                'model': model_type,
                'accuracy': metrics.get('accuracy', report.get('accuracy')),
                'precision': weighted.get('precision'),
                'recall': weighted.get('recall'),
                'f1_score': weighted.get('f1-score', weighted.get('f1_score')),
                'roc_auc': metrics.get('roc_auc'),
            })
    return pd.DataFrame(rows)

In [ ]:
# Dataset layout inspection
layout = inspect_dataset_structure(DATASET_ROOT)
print('Dataset root:', DATASET_ROOT.resolve())
for split, classes in layout.items():
    if classes is None:
        print(f'{split}: MISSING')
    elif not classes:
        print(f'{split}: EMPTY')
    else:
        print(f'{split}:')
        for class_name, count in classes.items():
            print(f'  - {class_name}: {count}')

In [ ]:
# Compare performance across models (if metrics exist)
comparison_df = build_comparison_frame(ARTIFACTS_ROOT)
if comparison_df.empty:
    print('No evaluation metrics found in artifacts directory yet.')
else:
    display(comparison_df)

In [ ]:
# Load a training history file if available
history_files = sorted((ARTIFACTS_ROOT / 'cnn').glob('history_*.npz'))
if history_files:
    history = load_history_npz(history_files[-1])
    print('Loaded history file:', history_files[-1])
    for metric_name, values in history.items():
        print(metric_name, len(values))
else:
    print('No history file found for CNN.')